In [27]:
# Test Preprocess: ERA5 Data Preprocessing Testing

#This notebook allows quick testing of the ERA5 preprocessing logic without running the full download workflow.

In [ ]:
## 1. Import Required Libraries

In [28]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import gc

In [ ]:
## 2. Set Up Test Parameters and Paths

In [29]:
# Configuración de prueba con archivos reales
START_DATE = "2023-01-01"
END_DATE = "2023-01-02"  # Solo 2 días para prueba rápida

start = pd.to_datetime(START_DATE)
end = pd.to_datetime(END_DATE)
year = start.year

# Usar directorios reales del workflow para input, pero output separado para pruebas
target_dir = "."  # Donde están los datos descargados
test_preprocessed_dir = "./test_preprocessed_data"  # Carpeta separada para pruebas

# Rutas
input_base = Path(target_dir) / f"{year}" / "01"  # Archivos mensuales descargados
output_base_dir = Path(test_preprocessed_dir) / "descompuestos" / "01"  # Output en carpeta de prueba

# Variables
surface_vars = ["tp", "sp", "e", "tcw"]
ml_vars = ["u", "v", "q"]

print(f"Test setup with real files. Input: {input_base}, Output: {output_base_dir}")
if not input_base.exists():
    print(f"Warning: Input directory {input_base} does not exist. Please run download first.")
else:
    print(f"Input files found: {list(input_base.glob('*.nc'))}")

Test setup with real files. Input: 2023/01, Output: test_preprocessed_data/descompuestos/01
Input files found: [PosixPath('2023/01/ERA5_2023-01_sp.nc'), PosixPath('2023/01/ERA5_2023-01_tp.nc'), PosixPath('2023/01/ERA5_2023-01_e.nc'), PosixPath('2023/01/ERA5_2023-01_ml_v.nc'), PosixPath('2023/01/ERA5_2023-01_ml_u.nc'), PosixPath('2023/01/ERA5_2023-01_ml_q.nc'), PosixPath('2023/01/ERA5_2023-01_tcw.nc')]


In [ ]:
## 3. Use Real Downloaded Files

In [30]:
# Verificar archivos reales
print("Checking for real input files...")
for var in surface_vars:
    filepath = input_base / f"ERA5_{year}-01_{var}.nc"
    if filepath.exists():
        print(f"✓ Surface file: {filepath}")
    else:
        print(f"✗ Missing surface file: {filepath}")

for var in ml_vars:
    filepath = input_base / f"ERA5_{year}-01_ml_{var}.nc"
    if filepath.exists():
        print(f"✓ ML file: {filepath}")
    else:
        print(f"✗ Missing ML file: {filepath}")

print("If files are missing, run the download step first.")

Checking for real input files...
✓ Surface file: 2023/01/ERA5_2023-01_tp.nc
✓ Surface file: 2023/01/ERA5_2023-01_sp.nc
✓ Surface file: 2023/01/ERA5_2023-01_e.nc
✓ Surface file: 2023/01/ERA5_2023-01_tcw.nc
✓ ML file: 2023/01/ERA5_2023-01_ml_u.nc
✓ ML file: 2023/01/ERA5_2023-01_ml_v.nc
✓ ML file: 2023/01/ERA5_2023-01_ml_q.nc
If files are missing, run the download step first.


In [ ]:
## 4. Test Preprocessing for Surface Variables

In [31]:
# Probar preprocesamiento de superficie con archivos reales
output_dir = output_base_dir
output_dir.mkdir(exist_ok=True, parents=True)

for var in surface_vars:
    archivo_mensual = input_base / f"ERA5_{year}-01_{var}.nc"
    if not archivo_mensual.exists():
        print(f"Archivo {archivo_mensual} no encontrado, saltando.")
        continue

    print(f"Procesando superficie: {var}")
    with xr.open_dataset(archivo_mensual, chunks={'time': 1}, decode_cf=True) as ds:
        ds = ds.drop_vars(['expver', 'number'], errors='ignore')
        if 'ps' in ds.data_vars:
            ds = ds.rename({'ps': 'sp'})
        if 'valid_time' in ds.coords:
            ds = ds.rename({'valid_time': 'time'})

        # Convertir tiempo a datetime si es cftime
        if hasattr(ds['time'].values[0], 'calendar'):
            ds['time'] = pd.to_datetime(ds['time'].values)

        time_coords = ds['time'].values
        unique_dates = np.unique(np.array([str(pd.Timestamp(t).date()) for t in time_coords]))

        for dia_str in unique_dates[:2]:  # Primeros 2 días para prueba
            ds_dia = ds.sel(time=ds['time'].dt.date == np.datetime64(dia_str))
            ds_dia = ds_dia.compute()

            for var_name in ds_dia.data_vars:
                ds_dia[var_name] = ds_dia[var_name].astype(np.float64)

            if 'model_level' in ds_dia.dims:
                ds_dia = ds_dia.transpose('time', 'level', 'latitude', 'longitude')
                ds_dia = ds_dia.assign_coords(longitude=(((ds_dia.longitude + 180) % 360) - 180))
            else:
                ds_dia = ds_dia.transpose('time', 'latitude', 'longitude')

            output_filename = f"ERA5_{dia_str}_{var}.nc"
            output_path = output_dir / output_filename
            ds_dia.to_netcdf(output_path)
            print(f"  Saved: {output_path}")

print("Surface preprocessing test complete.")

Procesando superficie: tp
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-01_tp.nc
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-02_tp.nc
Procesando superficie: sp
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-01_sp.nc
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-02_sp.nc
Procesando superficie: e
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-01_e.nc
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-02_e.nc
Procesando superficie: tcw
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-01_tcw.nc
  Saved: test_preprocessed_data/descompuestos/01/ERA5_2023-01-02_tcw.nc
Surface preprocessing test complete.


In [ ]:
## 5. Test Preprocessing for Model Level Variables

In [ ]:
# Probar preprocesamiento de ML con archivos reales (procesamiento diario completo)
# Ensure output_dir is Path object and set to test directory
output_dir = output_base_dir
output_dir.mkdir(exist_ok=True, parents=True)

for ml in ml_vars:
    archivo_mensual = input_base / f"ERA5_{year}-01_ml_{ml}.nc"
    if not archivo_mensual.exists():
        print(f"Archivo {archivo_mensual} no encontrado, saltando.")
        continue

    print(f"Procesando ML: {ml}")
    with xr.open_dataset(archivo_mensual, chunks={'time': 1}, decode_cf=True) as ds:
        ds = ds.drop_vars(['expver', 'number'], errors='ignore')
        if 'valid_time' in ds.coords:
            ds = ds.rename({'valid_time': 'time'})
        if 'ps' in ds.data_vars:
            ds = ds.rename({'ps': 'sp'})
        if 'model_level' in ds.dims:
            ds = ds.rename({'model_level': 'level'})

        # Convertir tiempo a datetime si es cftime
        if hasattr(ds['time'].values[0], 'calendar'):
            ds['time'] = pd.to_datetime(ds['time'].values)

        time_coords = ds['time'].values
        unique_dates = np.unique(np.array([str(pd.Timestamp(t).date()) for t in time_coords]))

        for dia_str in unique_dates[:2]:  # Primeros 2 días
            indices = np.where(np.array([str(pd.Timestamp(t).date()) for t in time_coords]) == dia_str)[0]
            if len(indices) == 0:
                continue

            # Procesar el día completo sin batching
            ds_dia = ds.isel(time=indices).compute()

            for var_name in ds_dia.data_vars:
                ds_dia[var_name] = ds_dia[var_name].astype(np.float64)

            # Transponer para consistencia
            if 'level' in ds_dia.dims:
                ds_dia = ds_dia.transpose('time', 'level', 'latitude', 'longitude')
                ds_dia = ds_dia.assign_coords(longitude=(((ds_dia.longitude + 180) % 360) - 180))
            else:
                ds_dia = ds_dia.transpose('time', 'latitude', 'longitude')

            output_filename = f"ERA5_{dia_str}_ml_{ml}.nc"
            output_path = output_dir / output_filename
            ds_dia.to_netcdf(output_path)

            print(f"  Día {dia_str} completado: {len(indices)} horas guardadas en {output_path}")

    del ds
    gc.collect()

print("ML preprocessing test complete (diario).")

Procesando ML: u
  Día 2023-01-01 completado: 24 horas guardadas en output_data/ERA5_2023-01-01_ml_u.nc
  Día 2023-01-02 completado: 24 horas guardadas en output_data/ERA5_2023-01-02_ml_u.nc
Procesando ML: v
  Día 2023-01-01 completado: 24 horas guardadas en output_data/ERA5_2023-01-01_ml_v.nc
  Día 2023-01-02 completado: 24 horas guardadas en output_data/ERA5_2023-01-02_ml_v.nc
Procesando ML: q
  Día 2023-01-01 completado: 24 horas guardadas en output_data/ERA5_2023-01-01_ml_q.nc
  Día 2023-01-02 completado: 24 horas guardadas en output_data/ERA5_2023-01-02_ml_q.nc
ML preprocessing test complete (diario).


In [ ]:
## 6. Debug and Fix Transpose Issues

In [38]:
# Depurar problemas de transpose
# Verificar dimensiones de archivos de salida
for f in output_dir.glob("*.nc"):
    print(f"File: {f.name}")
    with xr.open_dataset(f) as ds:
        print(f"  Dims: {dict(ds.dims)}")
        print(f"  Coords: {list(ds.coords)}")
        print(f"  Data vars: {list(ds.data_vars)}")
    print()

# Si hay errores, ajustar el transpose aquí
# Por ejemplo, asegurar que para ML sea ('time', 'level', 'latitude', 'longitude')

File: ERA5_2023-01-02_ml_q.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', 'longitude']
  Data vars: ['q']

File: ERA5_2023-01-02_ml_u.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', 'longitude']
  Data vars: ['u']

File: ERA5_2023-01-02_ml_v.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', 'longitude']
  Data vars: ['v']

File: ERA5_2023-01-01_ml_q.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', 'longitude']
  Data vars: ['q']

File: ERA5_2023-01-01_ml_v.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', 'longitude']
  Data vars: ['v']

File: ERA5_2023-01-01_ml_u.nc
  Dims: {'time': 24, 'level': 22, 'latitude': 181, 'longitude': 261}
  Coords: ['time', 'level', 'latitude', '

/tmp/ipykernel_136570/1164515755.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dims: {dict(ds.dims)}")
/tmp/ipykernel_136570/1164515755.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dims: {dict(ds.dims)}")
/tmp/ipykernel_136570/1164515755.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dims: {dict(ds.dims)}")
/tmp/ipykernel_136570/1164515755.py:6

In [ ]:
## 7. Validate Output Files

In [39]:
# Validar archivos de salida y comparar con ejemplo
print("Validating output files...")
example_dir = Path("/home/unal/wam/preprocessed_example")

for f in output_dir.glob("*.nc"):
    try:
        with xr.open_dataset(f) as ds:
            print(f"✓ {f.name}: shape={ds[list(ds.data_vars)[0]].shape}, dtype={ds[list(ds.data_vars)[0]].dtype}")
            print(f"  Dims: {list(ds.dims)}")
            print(f"  Coords: {list(ds.coords)}")
    except Exception as e:
        print(f"✗ {f.name}: {e}")

# Comparar con ejemplo si existe
if example_dir.exists():
    print("\nComparing with example files...")
    for example_file in example_dir.glob("*.nc"):
        test_file = output_dir / example_file.name
        if test_file.exists():
            with xr.open_dataset(example_file) as ex_ds, xr.open_dataset(test_file) as test_ds:
                print(f"Example {example_file.name}: {ex_ds[list(ex_ds.data_vars)[0]].shape}")
                print(f"Test {test_file.name}: {test_ds[list(test_ds.data_vars)[0]].shape}")
                if ex_ds[list(ex_ds.data_vars)[0]].shape == test_ds[list(test_ds.data_vars)[0]].shape:
                    print("  ✓ Shapes match")
                else:
                    print("  ✗ Shapes differ")
        else:
            print(f"Test file {test_file.name} not found")

print("Validation complete.")

Validating output files...
✓ ERA5_2023-01-02_ml_q.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']
  Coords: ['time', 'level', 'latitude', 'longitude']
✓ ERA5_2023-01-02_ml_u.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']
  Coords: ['time', 'level', 'latitude', 'longitude']
✓ ERA5_2023-01-02_ml_v.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']
  Coords: ['time', 'level', 'latitude', 'longitude']
✓ ERA5_2023-01-01_ml_q.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']
  Coords: ['time', 'level', 'latitude', 'longitude']
✓ ERA5_2023-01-01_ml_v.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']
  Coords: ['time', 'level', 'latitude', 'longitude']
✓ ERA5_2023-01-01_ml_u.nc: shape=(24, 22, 181, 261), dtype=float64
  Dims: ['time', 'level', 'latitude', 'longitude']


In [40]:
# Configuración de fechas y rutas
START_DATE = "2023-01-02"  # Fecha de inicio en formato YYYY-MM-DD
END_DATE = "2023-01-03"    # Fecha de fin en formato YYYY-MM-DD

# Parsear fechas
start = pd.to_datetime(START_DATE)
end = pd.to_datetime(END_DATE)

# Extraer año
year = start.year

# Generar rango de fechas
datelist = pd.date_range(START_DATE, END_DATE)
months = list(set([str(d.month).zfill(2) for d in datelist]))
months.sort()

# Diccionarios de días y fechas por mes
days_all = {}
dates = []
for month in months:
    month_days = datelist[datelist.month == int(month)]
    days_all[month] = [str(d.day).zfill(2) for d in month_days]
    month_str = month
    start_day = min(month_days).strftime("%d")
    end_day = max(month_days).strftime("%d")
    dates.append(f"{year}-{month}-{start_day}/to/{year}-{month}-{end_day}")

print(f"Período: {START_DATE} a {END_DATE}")
print(f"Meses a descargar: {months}")
print(f"Días por mes: {days_all}")

# Rutas generales
target_dir = "."
preprocessed_dir = "./preprocessed_data"
output_dir = "./output_data"
yaml_config_name = "workflow_config.yaml"

# Configuración de descarga
skip_exist = True
area = None  # None para global, o [N, W, S, E]
grid = [0.25, 0.25]

# Variables a descargar
ml_variables = {
    "u": 131,
    "v": 132,
    "q": 133,
}

surface_variables = {
    "tp": "total_precipitation",
    "e": "evaporation",
    "sp": "surface_pressure",
    "tcw": "total_column_water",
}

# Niveles de modelo
levels = "20/40/60/80/90/95/100/105/110/115/120/123/125/128/130/131/132/133/134/135/136/137"

# Tiempos (24 horas)
times = [f"{h:02d}:00" for h in range(24)]

print(f"\nConfiguraciones:")
print(f"- Directorio de salida preprocessado: {preprocessed_dir}")
print(f"- Directorio de output: {output_dir}")
print(f"- Variables ML: {list(ml_variables.keys())}")
print(f"- Variables superficie: {list(surface_variables.keys())}")

Período: 2023-01-02 a 2023-01-03
Meses a descargar: ['01']
Días por mes: {'01': ['02', '03']}

Configuraciones:
- Directorio de salida preprocessado: ./preprocessed_data
- Directorio de output: ./output_data
- Variables ML: ['u', 'v', 'q']
- Variables superficie: ['tp', 'e', 'sp', 'tcw']


In [41]:
print("="*80)
print("CREANDO ARCHIVO DE CONFIGURACIÓN YAML")
print("="*80)

# Crear estructura YAML manualmente para asegurar que las fechas estén entre comillas
yaml_path = Path(yaml_config_name)

yaml_text = f"""# Output
preprocessed_data_folder: {preprocessed_dir}
output_folder: {output_dir}
output_frequency: "1d"

# Preprocessing
filename_template: "{'test_preprocessed_data/descompuestos/{month:02d}/ERA5_{year}-{month:02d}-{day:02d}{levtype}_{variable}.nc' }"
preprocess_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
preprocess_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
level_type: model_levels
levels: [20,40,60,80,90,95,100,105,110,115,120,123,125,128,130,131,132,133,134,135,136,137]
input_frequency: "1h"

# Tracking
tracking_direction: backward
tracking_domain: null
periodic_boundary: False
tracking_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
tracking_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
timestep: 600
kvf: 3

# Tagging
tagging_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
tagging_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
restart: False
tagging_region: shapes/sinu.nc
target_frequency: "10min"
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_text)

print(f"✓ Archivo YAML creado: {yaml_path}")
print(f"\nContenido del archivo YAML:")
print("-" * 80)
with open(yaml_path, 'r') as f:
    print(f.read())
print("-" * 80)

CREANDO ARCHIVO DE CONFIGURACIÓN YAML
✓ Archivo YAML creado: workflow_config.yaml

Contenido del archivo YAML:
--------------------------------------------------------------------------------
# Output
preprocessed_data_folder: ./preprocessed_data
output_folder: ./output_data
output_frequency: "1d"

# Preprocessing
filename_template: "test_preprocessed_data/descompuestos/{month:02d}/ERA5_{year}-{month:02d}-{day:02d}{levtype}_{variable}.nc"
preprocess_start_date: "2023-01-02T00:00"
preprocess_end_date: "2023-01-03T00:00"
level_type: model_levels
levels: [20,40,60,80,90,95,100,105,110,115,120,123,125,128,130,131,132,133,134,135,136,137]
input_frequency: "1h"

# Tracking
tracking_direction: backward
tracking_domain: null
periodic_boundary: False
tracking_start_date: "2023-01-02T00:00"
tracking_end_date: "2023-01-03T00:00"
timestep: 600
kvf: 3

# Tagging
tagging_start_date: "2023-01-02T00:00"
tagging_end_date: "2023-01-03T00:00"
restart: False
tagging_region: shapes/sinu.nc
target_frequency